In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from src.utils import Utility
from src.vector_store import VectorStoreManager

nist_url = "https://nvlpubs.nist.gov/nistpubs/ai/NIST.AI.100-1.pdf"
eu_url = "https://spiderhub.cedia.edu.ec/en/documents/284/export-pdf"

nist_local_path = Utility.download_regulatory_pdf(nist_url, "NIST_AI_RMF.pdf")
eu_local_path = Utility.download_regulatory_pdf(eu_url, "EU_AI_Act.pdf")

nist_nodes = Utility.extract_metadata_chunks(nist_local_path, "NIST_AI_RMF.pdf")
eu_nodes = Utility.extract_metadata_chunks(eu_local_path, "EU_AI_Act.pdf")

all_regulatory_nodes = nist_nodes + eu_nodes
vector_db = VectorStoreManager.create_local_vector_store(all_regulatory_nodes, "data/regulatory_faiss")



In [ ]:
from src.vector_store import VectorStoreManager

loaded_db = VectorStoreManager.load_local_vector_store("data/regulatory_faiss")
query = "What are the risk management requirements for AI systems?"

docs = loaded_db.similarity_search(query, k = 2)


In [ ]:
from src.router import GatekeeperRouter

clear_query = "What is the official risk management framework of NIST?"
route_result_1 = GatekeeperRouter.calculate_uncertainty_route(clear_query, loaded_db)

print("\n--- 🧭 ROUTER TELEMETRY: TEST 1 ---")
print(f"📊 Calculated Entropy : {route_result_1['entropy']}")
print(f"🔀 Assigned Route     : {route_result_1['selected_route']}")
print(f"🛠️ Action Triggered    : {route_result_1['action_code']}\n")

complex_query = "Compare EU Act classification with NIST and detect hidden compliance holes in high risk systems"
route_result_2 = GatekeeperRouter.calculate_uncertainty_route(complex_query, loaded_db)

print("\n--- 🧭 ROUTER TELEMETRY: TEST 2 ---")
print(f"📊 Calculated Entropy : {route_result_2['entropy']}")
print(f"🔀 Assigned Route     : {route_result_2['selected_route']}")
print(f"🛠️ Action Triggered    : {route_result_2['action_code']}")




In [ ]:
%reload_ext autoreload
%autoreload 2

from src.pipeline import ComplianceEngine

pipeline = ComplianceEngine(vector_db=loaded_db)
user_query = "We are deploying a generative AI system for hiring that utilizes automated biometric analysis. NIST AI RMF strongly recommends user-centric flexibility and voluntary compliance metrics, but the EU AI Act classifies high-risk AI strictly under mandatory conformity assessments. There is a direct friction here regarding deployment speed vs strict adherence. Evaluate the multi-agent compliance risks, look for contradictions in our retrieval documents, and provide an executive mitigation strategy."
result = pipeline.run(user_query)

print(f"Route Taken  : {result['route_taken']}")
print(f"Time Elapsed : {result['execution_time']} seconds\n")
print(result['output'])

